In [ ]:
# Lab 9, notebook 2: add an AI label to every clean ticket, paying for each distinct text only once.
# Attached lakehouse: freshcart_lake. It calls the class model as you (your sign-in, no key).

### Settings: the model, the prompt version and the price

In [ ]:
import hashlib, json, time, urllib.error, urllib.request
from pyspark.sql import functions as F
from delta.tables import DeltaTable

FOUNDRY = "PASTE-THE-ADDRESS-YOUR-TRAINER-GIVES-YOU"   # your trainer gives you this address
MODEL = "gpt-4.1-mini"
PROMPT_VERSION = "v1"           # change the prompt? change this, and every text is labelled again
PRICE_PER_MILLION = {"input": 0.40, "output": 1.60}   # US dollars, gpt-4.1-mini list price; check yours
CATEGORIES = ["missing_item", "late_delivery", "payment", "spillage", "other", "account", "food_safety", "wrong_item"]
PROMPT = ("You label FreshCart support tickets. Answer with exactly one of these categories and nothing else: "
          + ", ".join(CATEGORIES) + ".")

# Lab 10: a longer prompt someone wrote "to make the labels better". Switch it on to see what it does to cost.
USE_PROMPT_V2 = False
EXAMPLES = [("my parcel came two hours after the slot", "late_delivery"), ("the rider was very late", "late_delivery"),
            ("delivery took forever", "late_delivery"), ("driver went to the wrong building", "late_delivery"),
            ("one dish was not in the bag", "missing_item"), ("the drinks were missing", "missing_item"),
            ("half my order never came", "missing_item"), ("no cutlery and no raita", "missing_item"),
            ("I was charged two times", "payment"), ("the refund still has not come", "payment"),
            ("card declined but money taken", "payment"), ("promo code did not apply", "payment"),
            ("curry leaked everywhere", "spillage"), ("soup spilled in the bag", "spillage"),
            ("the box was soaked", "spillage"), ("drink lid came off", "spillage"),
            ("cannot log in to the app", "account"), ("app crashes when I reorder", "account"),
            ("I want to change my phone number", "account"), ("my address will not save", "account"),
            ("the chicken smelled off", "food_safety"), ("found a hair in the food", "food_safety"),
            ("food was undercooked", "food_safety"), ("I got sick after eating", "food_safety"),
            ("got the veg one instead of chicken", "wrong_item"), ("wrong size pizza", "wrong_item"),
            ("this is someone else's order", "wrong_item"), ("they sent the wrong flavour", "wrong_item"),
            ("food was cold when it came", "other"), ("packaging was torn", "other"),
            ("rider was rude", "other"), ("just want to give feedback", "other"),
            ("portion was smaller than before", "other"), ("the bag was dirty", "other"),
            ("please call me back", "other"), ("how do I tip the rider", "other"),
            ("order arrived late again", "late_delivery"), ("item missing from my bag", "missing_item"),
            ("double charge on my card", "payment"), ("whole order spilled", "spillage")]
PROMPT_V2 = (PROMPT + " Definitions: late_delivery means the order or rider was late or could not find the address; "
             "missing_item means something ordered did not arrive; payment means money was taken wrongly or a refund "
             "is late; spillage means food or drink leaked; account means the app or the customer's account; "
             "food_safety means the food may be unsafe; wrong_item means a different item came; other is anything "
             "else. Examples:\n" + "\n".join(f"{text} -> {label}" for text, label in EXAMPLES))
if USE_PROMPT_V2:
    PROMPT, PROMPT_VERSION = PROMPT_V2, "v2"
started = time.time()

### Ask the model about one text

In [ ]:
def label(text):
    """One call: the category, and the tokens it cost."""
    token = notebookutils.credentials.getToken("https://cognitiveservices.azure.com")
    body = {"messages": [{"role": "system", "content": PROMPT}, {"role": "user", "content": text}],
            "temperature": 0, "max_tokens": 10}
    request = urllib.request.Request(f"{FOUNDRY}/openai/deployments/{MODEL}/chat/completions?api-version=2024-10-21",
                                     json.dumps(body).encode(),
                                     {"Authorization": "Bearer " + token, "Content-Type": "application/json"})
    for attempt in range(3):                       # a busy model (429) or a hiccup: wait and try again, twice
        try:
            answer = json.load(urllib.request.urlopen(request, timeout=60))
            break
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as error:
            if attempt == 2 or getattr(error, "code", 429) not in (429, 500, 502, 503, 504):
                raise
            time.sleep(10 * (attempt + 1))
    word = answer["choices"][0]["message"]["content"].strip().lower()
    return {"label": word if word in CATEGORIES else "other",          # an answer off the list counts as other
            "off_list": word not in CATEGORIES,
            "tokens_in": answer["usage"]["prompt_tokens"], "tokens_out": answer["usage"]["completion_tokens"]}

print(label("my rice was missing"))

### Label only the texts we have not labelled before (the cache)

In [ ]:
def cache_key(text):
    """Same text, same model, same prompt version: same key, so no second call."""
    return hashlib.sha256(f"{MODEL}|{PROMPT_VERSION}|{text}".encode()).hexdigest()

texts = [r.text for r in spark.read.table("tickets_clean").select("text").distinct().collect() if r.text]
known = set()
if spark.catalog.tableExists("ticket_labels"):
    known = {r.key for r in spark.read.table("ticket_labels").select("key").collect()}
todo = [t for t in texts if cache_key(t) not in known]
print(f"distinct texts: {len(texts)} | already labelled: {len(texts) - len(todo)} | to label now: {len(todo)}")

def save(batch):
    """Add labels to the cache. Called every few labels, so a failure keeps everything already paid for."""
    if not batch:
        return
    frame = spark.createDataFrame(batch)
    if not spark.catalog.tableExists("ticket_labels"):
        frame.limit(0).write.format("delta").saveAsTable("ticket_labels")
    (DeltaTable.forName(spark, "ticket_labels").alias("t").merge(frame.alias("s"), "t.key = s.key")
        .whenNotMatchedInsertAll().execute())

new_labels, unsaved = [], []
try:
    for text in todo:
        result = label(text)
        row = {"key": cache_key(text), "text": text, "model": MODEL, "prompt_version": PROMPT_VERSION,
               "labelled_at": time.strftime("%Y-%m-%d %H:%M:%S"), **result}
        new_labels.append(row)
        unsaved.append(row)
        if len(unsaved) == 5:
            save(unsaved)
            unsaved = []
finally:
    save(unsaved)                                  # even if a call failed: keep the labels we paid for
if not spark.catalog.tableExists("ticket_labels"):        # an empty lakehouse: nothing labelled yet, so stop here
    notebookutils.notebook.exit("no tickets to label yet")
display(spark.read.table("ticket_labels").filter(F.col("prompt_version") == PROMPT_VERSION)
        .select("text", "label", "off_list", "model", "prompt_version"))

### Join the labels onto every clean ticket, with where each label came from

In [ ]:
labels = (spark.read.table("ticket_labels").filter((F.col("model") == MODEL) & (F.col("prompt_version") == PROMPT_VERSION))
          .select("text", F.col("label").alias("ai_category"), F.col("model").alias("ai_model"),
                  F.col("prompt_version").alias("ai_prompt_version"), F.col("labelled_at").alias("ai_labelled_at")))
enriched = (spark.read.table("tickets_clean").join(labels, "text", "left")
            .withColumn("categories_agree", F.col("category") == F.col("ai_category")))
enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("tickets_enriched")
rows = spark.read.table("tickets_enriched").count()
agree = spark.read.table("tickets_enriched").filter("categories_agree").count()
print(f"tickets enriched: {rows} | recorded category agrees with the AI label: {agree} ({agree / max(rows, 1):.1%})")
display(spark.read.table("tickets_enriched").groupBy("text", "ai_category", "category").count()
        .orderBy("text", F.desc("count")).limit(16))

### What it cost, per run and per row

In [ ]:
tokens_in = sum(l["tokens_in"] for l in new_labels)
tokens_out = sum(l["tokens_out"] for l in new_labels)
cost = tokens_in / 1e6 * PRICE_PER_MILLION["input"] + tokens_out / 1e6 * PRICE_PER_MILLION["output"]
# The average cost of one call, from every label of this model and prompt version (also on a run with no calls).
avg = (spark.read.table("ticket_labels").filter((F.col("model") == MODEL) & (F.col("prompt_version") == PROMPT_VERSION))
       .agg(F.avg("tokens_in").alias("i"), F.avg("tokens_out").alias("o")).first())
per_call = ((avg.i or 0) / 1e6 * PRICE_PER_MILLION["input"] + (avg.o or 0) / 1e6 * PRICE_PER_MILLION["output"])
run = {"run_at": time.strftime("%Y-%m-%d %H:%M:%S"), "model": MODEL, "prompt_version": PROMPT_VERSION,
       "texts": len(texts), "model_calls": len(new_labels), "cache_hits": len(texts) - len(new_labels),
       "tokens_in": tokens_in, "tokens_out": tokens_out, "cost_usd": float(round(cost, 6)),
       "cost_per_row_usd": cost / rows if rows else 0.0, "rows": rows,
       "one_call_per_row_usd": float(round(per_call * rows, 2)),     # what calling the model for every ticket would cost
       "seconds": round(time.time() - started, 1)}
spark.createDataFrame([run]).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("enrich_log")
print(json.dumps(run, indent=1))
display(spark.read.table("enrich_log").orderBy(F.desc("run_at")))